In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

project_dir = Path.cwd()
if not (project_dir / "src").is_dir():
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from src.face_detection import FaceDetector
from src.extract_signal import extract_rgb_traces, pos_algorithm
from src.features import compute_features
from src.filters import bandpass, detrend

video_extensions = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".mpeg", ".mpg"}
examples = {}
for label in ("real", "fake"):
    folder = project_dir / "data" / label
    videos = sorted(p for p in folder.rglob("*") if p.suffix.lower() in video_extensions)
    if not videos:
        raise FileNotFoundError(f"Add a video to {folder} before running this notebook.")
    examples[label] = videos[0]

examples


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, (label, video_path) in zip(axes, examples.items()):
    capture = cv2.VideoCapture(str(video_path))
    ok, frame_bgr = capture.read()
    capture.release()
    if not ok:
        raise ValueError(f"Could not read the first frame of {video_path}")
    detector = FaceDetector()
    try:
        masks = detector.get_roi_masks(frame_bgr)
    finally:
        detector.close()
    if masks is None:
        raise ValueError(f"No face found in the first frame of {video_path}")
    overlay = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB).copy()
    for mask in masks.values():
        overlay[mask > 0] = (0, 255, 0)
    original = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    overlay = (0.35 * overlay + 0.65 * original).astype(np.uint8)
    axis.imshow(overlay)
    axis.set_title(f"{label}: {video_path.name}")
    axis.axis("off")
plt.tight_layout()


In [ ]:
traces_by_label = {}
fps_by_label = {}
for label, video_path in examples.items():
    traces_by_label[label], fps_by_label[label] = extract_rgb_traces(video_path)

fig, axes = plt.subplots(3, 2, figsize=(13, 9))
for column, (label, traces) in enumerate(traces_by_label.items()):
    fps = fps_by_label[label]
    for row, (roi, rgb_trace) in enumerate(traces.items()):
        count = min(len(rgb_trace), int(5 * fps))
        times = np.arange(count) / fps
        for channel, color in enumerate(("red", "green", "blue")):
            axes[row, column].plot(times, rgb_trace[:count, channel], color=color, label=color)
        axes[row, column].set_title(f"{label} — {roi}")
        axes[row, column].set_xlabel("Time (seconds)")
        axes[row, column].set_ylabel("Mean color value")
        if row == 0:
            axes[row, column].legend()
plt.tight_layout()


In [ ]:
from scipy.signal import welch

filtered_by_label = {}
features_by_label = {}
names_by_label = {}
for label, traces in traces_by_label.items():
    fps = fps_by_label[label]
    filtered = {
        roi: bandpass(detrend(pos_algorithm(rgb_trace, fps)), fps)
        for roi, rgb_trace in traces.items()
    }
    filtered_by_label[label] = filtered
    features_by_label[label], names_by_label[label] = compute_features(filtered, fps)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for column, (label, roi_signals) in enumerate(filtered_by_label.items()):
    fps = fps_by_label[label]
    for row, (roi, signal) in enumerate(roi_signals.items()):
        time = np.arange(len(signal)) / fps
        axes[row, column * 2].plot(time, signal, linewidth=0.8)
        axes[row, column * 2].set_title(f"{label} — {roi} pulse")
        axes[row, column * 2].set_xlabel("Time (seconds)")
        axes[row, column * 2].set_ylabel("Filtered signal")

        frequencies, power = welch(signal, fs=fps, nperseg=min(len(signal), max(512, int(8 * fps))))
        in_band = (frequencies >= 0.7) & (frequencies <= min(4.0, fps / 2))
        axes[row, column * 2 + 1].plot(frequencies[in_band], power[in_band])
        feature_names = names_by_label[label]
        feature_values = features_by_label[label]
        peak_hz = feature_values[feature_names.index(f"{roi}_dominant_frequency_hz")]
        bpm = feature_values[feature_names.index(f"{roi}_estimated_bpm")]
        if peak_hz > 0:
            axes[row, column * 2 + 1].axvline(peak_hz, color="red", linestyle="--", label=f"{bpm:.0f} BPM")
            axes[row, column * 2 + 1].legend()
        axes[row, column * 2 + 1].set_title(f"{label} — {roi} spectrum")
        axes[row, column * 2 + 1].set_xlabel("Frequency (Hz)")
        axes[row, column * 2 + 1].set_ylabel("Power")
plt.tight_layout()


In [ ]:
feature_comparison = {
    feature_name: {
        label: float(values[index])
        for label, values in features_by_label.items()
    }
    for index, feature_name in enumerate(names_by_label["real"])
}
feature_comparison
